# Organisation manuelle du dataset SIGHT

Ce notebook sert de guide pratique pour ranger SIGHT dans une version **processed** simple et utile pour Smart Teacher.

Objectif:
- garder les fichiers bruts intacts
- relier les commentaires aux vidéos
- préparer un tableau unifié pour l'entraînement
- garder une option RAG si on veut chercher dans les transcriptions

## 1. Dossiers vraiment utiles

Pour un usage simple, il suffit de garder 4 dossiers sources et 1 dossier de sortie:

- `comments/` : commentaires étudiants bruts
- `transcripts/` : texte des vidéos
- `metadata/` : infos sur la vidéo et la playlist
- `annotations/` : labels ou colonnes d'annotation
- `processed/` : fichiers dérivés uniquement

On travaille ensuite surtout avec `sight_dataset.csv` pour les jointures, puis on relie les autres fichiers autour de `video_id` et `comment_id`.

Ce qui est important:
- ne pas modifier les fichiers bruts
- ne pas dupliquer les transcriptions dans chaque ligne commentaire
- créer les fichiers de sortie seulement dans `processed/`


In [1]:
from pathlib import Path
import json
import csv
import pandas as pd

base_dir = Path('..').resolve()
raw_dirs = ['comments', 'transcripts', 'metadata', 'annotations']

print('Base:', base_dir)
for name in raw_dirs:
    path = base_dir / name
    print(name, '->', 'OK' if path.exists() else 'MISSING')


Base: C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data
comments -> OK
transcripts -> OK
metadata -> OK
annotations -> OK


## 2. Clés de liaison

On travaille maintenant directement avec `data/annotations/sight_dataset.csv`, pas avec le petit échantillon.

Les deux clés vraiment utiles sont:

- `video_id` : relie `metadata/` et `transcripts/`
- `comment_id` : relie les annotations entre elles et au commentaire exact

Les jointures à faire sont simples:

- `sight_dataset.csv` + `metadata/*.json` sur `video_id`
- `sight_dataset.csv` + `transcripts/*.txt` sur `video_id`
- les colonnes d'annotations sur `comment_id`
- si on veut une version finale ML, on garde une ligne par commentaire avec le texte, le cours, le transcript lié et le label

Règle simple:
- on utilise `video_id` pour joindre la vidéo au texte du cours
- on utilise `comment_id` pour joindre le commentaire à ses labels

Donc, pour organiser manuellement le dataset, il faut construire une ligne unifiée par commentaire:
- texte du commentaire
- `video_id`
- titre de la vidéo
- playlist / cours
- label(s)
- éventuellement le transcript associé


In [8]:
# SIGHT uses a space-delimited text file with quoted fields.
annotations_path = base_dir / 'annotations' / 'sight_dataset.csv'
metadata_dir = base_dir / 'metadata'

if annotations_path.exists():
    with annotations_path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle, delimiter=' ', quotechar='"', skipinitialspace=True)
        rows = list(reader)
    annotations_df = pd.DataFrame(rows[1:], columns=rows[0])

    print('Annotation columns:')
    print(list(annotations_df.columns))
    print('\nFirst rows:')
    display(annotations_df.head(3))
else:
    print('sight_dataset.csv not found')

print('\nMetadata files (first 5):')
for path in sorted(metadata_dir.glob('*.json'))[:5]:
    print(path.name)


Annotation columns:
['comment', 'video_id', 'video_name', 'playlist_name', 'comment_id', 'annotator_openaichat_v0_setup', 'annotator_openaichat_v8_confusion', 'annotator_openaichat_v74_pedagogy']

First rows:


,comment,video_id,video_name,playlist_name,comment_id,annotator_openaichat_v0_setup,annotator_openaichat_v8_confusion,annotator_openaichat_v74_pedagogy
0,Jerison rocks!,HgEqXhsIq_g,"Lec 29 | MIT 18.01 Single Variable Calculus, F...","MIT 18.01 Single Variable Calculus, Fall 2006",7134.0,0.0,0.0,0.0
1,46:27 Top 10 saddest anime betrayals,9FLItlbBUPY,Lec 2: Determinants; cross product | MIT 18.02...,"MIT 18.02 Multivariable Calculus, Fall 2007",1270.0,0.0,0.0,0.0
2,"moises cañas, mira como en el mit si responden...",ryLdyDrBfvI,"Lec 2 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",13959.0,0.0,0.0,0.0



Metadata files (first 5):
-5JSjp4969E.json
-AVYY53jL5w.json
-bGpKfRRqhU.json
-Dpkjz3TdpA.json
-EBlBKsfN08.json


## 7. Utiliser SIGHT pour détecter la confusion dans Smart Teacher

SIGHT sert surtout ici à entraîner ou valider un classifieur de confusion.

Dans notre projet, le flux logique est:

- l'étudiant écrit un message ou parle
- le texte est transcrit si besoin
- on calcule un score de confusion sur ce texte
- si la confusion est forte, on passe en mode `CLARIFICATION`
- sinon, on répond normalement avec le prompt standard

Important:
- `sight_dataset.csv` sert pour l'entraînement offline
- les commentaires servent d'entrée texte
- le label confusion peut être binaire pour commencer: `1 = confusion`, `0 = pas confusion`
- si tu veux plus de qualité, tu peux garder aussi le niveau de confiance ou le type de label

Dans le projet Smart Teacher, cela correspond à la logique déjà présente dans le pipeline de dialogue et dans la transition vers `CLARIFICATION`.


In [10]:
if 'annotations_df' in globals():
    confusion_df = annotations_df[['comment', 'video_id', 'video_name', 'playlist_name', 'annotator_openaichat_v8_confusion']].copy()
    confusion_df['confusion_label'] = pd.to_numeric(
        confusion_df['annotator_openaichat_v8_confusion'], errors='coerce'
    ).fillna(0).astype(int)

    print('Confusion label counts:')
    print(confusion_df['confusion_label'].value_counts())

    print('\nExamples of confused comments:')
    display(confusion_df[confusion_df['confusion_label'] == 1].head(10))
else:
    print('annotations_df not loaded')


Confusion label counts:
confusion_label
0    13242
1     2542
Name: count, dtype: int64

Examples of confused comments:


,comment,video_id,video_name,playlist_name,annotator_openaichat_v8_confusion,confusion_label
3,"i'm still a little confused, what is u naught ...",13r9QY6cmjc,22. Diagonalization and Powers of A,"MIT 18.06 Linear Algebra, Spring 2005",1.0,1
7,"So, can someone explain (or point in the right...",Z_5uLqcwDgM,Lecture 10: Survey of Difficulties with Ax = b,"MIT 18.065 Matrix Methods in Data Analysis, Si...",1.0,1
9,How can I find the proof of Eckart-Young theor...,Y4f7K9XF04k,7. Eckart-Young: The Closest Rank k Matrix to A,"MIT 18.065 Matrix Methods in Data Analysis, Si...",1.0,1
17,06:58 Anybody knows that why the dimension of ...,2IdtqGM6KWU,11. Matrix Spaces; Rank 1; Small World Graphs,"MIT 18.06 Linear Algebra, Spring 2005",1.0,1
20,50.00 wrong solution?,sRIDVAcoG5A,"Lec 13 | MIT 18.01 Single Variable Calculus, F...","MIT 18.01 Single Variable Calculus, Fall 2006",1.0,1
23,so there is really a mistake at 33:11... it ha...,YN7k_bXXggY,"Lec 12 | MIT 18.01 Single Variable Calculus, F...","MIT 18.01 Single Variable Calculus, Fall 2006",1.0,1
24,Thanks for the great video! One question: what...,a1ZCeFpeW0o,20. Principal Component Analysis (cont.),"MIT 18.650 Statistics for Applications, Fall 2016",1.0,1
32,This guy screwed up at 26:27 by by using d(e^w...,9v25gg2qJYE,"Lec 6 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",1.0,1
34,I didn't get the 'mathemagic' transformation o...,9BYsNpTCZGg,Lecture 17: Rapidly Decreasing Singular Values,"MIT 18.065 Matrix Methods in Data Analysis, Si...",1.0,1
35,I paused the video when he wrote pi/2 and solv...,YP_B0AapU0c,Lec 16: Double integrals | MIT 18.02 Multivari...,"MIT 18.02 Multivariable Calculus, Fall 2007",1.0,1


## 8. Préparer le dataset binaire prêt pour l'entraînement

Ici on transforme SIGHT en un vrai dataset de classification.

But:
- entrée = texte du commentaire
- sortie = `confusion_label`
- split = par `video_id` pour éviter la fuite de données

C'est ce fichier qui servira ensuite à entraîner un modèle BERT ou XLM-R.


In [11]:
import random

if 'annotations_df' in globals():
    binary_df = annotations_df[['comment', 'video_id', 'video_name', 'playlist_name', 'comment_id', 'annotator_openaichat_v8_confusion']].copy()
    binary_df = binary_df.rename(columns={
        'comment': 'comment_text',
        'video_name': 'video_title',
        'playlist_name': 'playlist_title',
        'annotator_openaichat_v8_confusion': 'confusion_source_label',
    })

    binary_df['comment_id'] = binary_df['comment_id'].astype(str).str.replace(r'\.0$', '', regex=True)
    binary_df['confusion_label'] = pd.to_numeric(binary_df['confusion_source_label'], errors='coerce').fillna(0).astype(int)
    binary_df['label_source'] = 'annotator_openaichat_v8_confusion'

    video_ids = sorted(binary_df['video_id'].dropna().astype(str).unique())
    rng = random.Random(42)
    rng.shuffle(video_ids)

    n_videos = len(video_ids)
    train_end = max(int(n_videos * 0.8), 1) if n_videos else 0
    val_end = max(int(n_videos * 0.9), train_end) if n_videos else 0

    split_map = {}
    for index, video_id in enumerate(video_ids):
        if index < train_end:
            split_map[video_id] = 'train'
        elif index < val_end:
            split_map[video_id] = 'validation'
        else:
            split_map[video_id] = 'test'

    binary_df['video_id'] = binary_df['video_id'].astype(str)
    binary_df['split'] = binary_df['video_id'].map(split_map)
    binary_df['model_text'] = binary_df['comment_text']

    output_dir = base_dir / 'processed'
    output_dir.mkdir(parents=True, exist_ok=True)

    train_df = binary_df[binary_df['split'] == 'train'].copy()
    val_df = binary_df[binary_df['split'] == 'validation'].copy()
    test_df = binary_df[binary_df['split'] == 'test'].copy()

    binary_path = output_dir / 'sight_confusion_binary.csv'
    train_path = output_dir / 'sight_confusion_train.csv'
    val_path = output_dir / 'sight_confusion_validation.csv'
    test_path = output_dir / 'sight_confusion_test.csv'
    summary_path = output_dir / 'sight_confusion_summary.json'

    binary_df.to_csv(binary_path, index=False)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    summary = {
        'rows': int(len(binary_df)),
        'videos': int(n_videos),
        'confusion_counts': binary_df['confusion_label'].value_counts().to_dict(),
        'split_counts': binary_df['split'].value_counts().to_dict(),
        'output_files': {
            'binary': str(binary_path),
            'train': str(train_path),
            'validation': str(val_path),
            'test': str(test_path),
        },
    }

    with summary_path.open('w', encoding='utf-8') as handle:
        json.dump(summary, handle, ensure_ascii=False, indent=2)

    print('Binary confusion dataset created.')
    print('Rows:', len(binary_df))
    print('Videos:', n_videos)
    print('\nConfusion counts:')
    print(binary_df['confusion_label'].value_counts())
    print('\nSplit counts:')
    print(binary_df['split'].value_counts())
    print('\nSaved files:')
    print(binary_path)
    print(train_path)
    print(val_path)
    print(test_path)
    print(summary_path)
else:
    print('annotations_df not loaded')


Binary confusion dataset created.
Rows: 15784
Videos: 272

Confusion counts:
confusion_label
0    13242
1     2542
Name: count, dtype: int64

Split counts:
split
train         11466
test           2687
validation     1631
Name: count, dtype: int64

Saved files:
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_confusion_binary.csv
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_confusion_train.csv
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_confusion_validation.csv
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_confusion_test.csv
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_confusion_summary.json


In [12]:
# Display the first few rows of the training set
if 'train_df' in globals():
    print('Training set preview:')
    display(train_df.head(5))



Training set preview:


,comment_text,video_id,video_title,playlist_title,comment_id,confusion_source_label,confusion_label,label_source,split,model_text
0,Jerison rocks!,HgEqXhsIq_g,"Lec 29 | MIT 18.01 Single Variable Calculus, F...","MIT 18.01 Single Variable Calculus, Fall 2006",7134,0.0,0,annotator_openaichat_v8_confusion,train,Jerison rocks!
2,"moises cañas, mira como en el mit si responden...",ryLdyDrBfvI,"Lec 2 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",13959,0.0,0,annotator_openaichat_v8_confusion,train,"moises cañas, mira como en el mit si responden..."
3,"i'm still a little confused, what is u naught ...",13r9QY6cmjc,22. Diagonalization and Powers of A,"MIT 18.06 Linear Algebra, Spring 2005",13518,1.0,1,annotator_openaichat_v8_confusion,train,"i'm still a little confused, what is u naught ..."
6,Pen di siri calculus de andar poori dunya basi...,BSAA0akmPEU,"Lec 9 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",8160,0.0,0,annotator_openaichat_v8_confusion,train,Pen di siri calculus de andar poori dunya basi...
7,"So, can someone explain (or point in the right...",Z_5uLqcwDgM,Lecture 10: Survey of Difficulties with Ax = b,"MIT 18.065 Matrix Methods in Data Analysis, Si...",8807,1.0,1,annotator_openaichat_v8_confusion,train,"So, can someone explain (or point in the right..."


## 9. Format direct pour BERT / XLM-R

Pour entraîner un premier modèle, on garde une entrée très simple:

- `text` = commentaire étudiant
- `label` = `confusion_label`
- `split` = train / validation / test

Pourquoi c'est le bon point de départ:
- le modèle apprend juste à reconnaître la confusion dans le texte
- on évite d'ajouter trop de variables au début
- ce format marche pour BERT, XLM-R et d'autres classifieurs texte

Plus tard, si tu veux améliorer la précision, tu pourras ajouter le transcript comme contexte.


In [13]:
if 'binary_df' not in globals() and 'annotations_df' in globals():
    binary_df = annotations_df[['comment', 'video_id', 'video_name', 'playlist_name', 'comment_id', 'annotator_openaichat_v8_confusion']].copy()
    binary_df = binary_df.rename(columns={
        'comment': 'comment_text',
        'video_name': 'video_title',
        'playlist_name': 'playlist_title',
        'annotator_openaichat_v8_confusion': 'confusion_source_label',
    })
    binary_df['comment_id'] = binary_df['comment_id'].astype(str).str.replace(r'\.0$', '', regex=True)
    binary_df['confusion_label'] = pd.to_numeric(binary_df['confusion_source_label'], errors='coerce').fillna(0).astype(int)

if 'binary_df' in globals():
    bert_df = binary_df[['comment_id', 'video_id', 'comment_text', 'video_title', 'playlist_title', 'confusion_label', 'split']].copy()
    bert_df = bert_df.rename(columns={
        'comment_text': 'text',
        'confusion_label': 'label',
    })
    bert_df['text'] = bert_df['text'].fillna('').astype(str).str.strip()
    bert_df['label'] = bert_df['label'].astype(int)

    bert_path = base_dir / 'processed' / 'sight_bert_format.csv'
    bert_df.to_csv(bert_path, index=False)

    print('BERT/XLM-R format created.')
    print('Rows:', len(bert_df))
    print('Columns:', list(bert_df.columns))
    print('\nLabel counts:')
    print(bert_df['label'].value_counts())
    print('\nSaved file:')
    print(bert_path)
    print('\nPreview:')
    display(bert_df.head(10))
else:
    print('binary_df not available')


BERT/XLM-R format created.
Rows: 15784
Columns: ['comment_id', 'video_id', 'text', 'video_title', 'playlist_title', 'label', 'split']

Label counts:
label
0    13242
1     2542
Name: count, dtype: int64

Saved file:
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_bert_format.csv

Preview:


,comment_id,video_id,text,video_title,playlist_title,label,split
0,7134,HgEqXhsIq_g,Jerison rocks!,"Lec 29 | MIT 18.01 Single Variable Calculus, F...","MIT 18.01 Single Variable Calculus, Fall 2006",0,train
1,1270,9FLItlbBUPY,46:27 Top 10 saddest anime betrayals,Lec 2: Determinants; cross product | MIT 18.02...,"MIT 18.02 Multivariable Calculus, Fall 2007",0,test
2,13959,ryLdyDrBfvI,"moises cañas, mira como en el mit si responden...","Lec 2 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",0,train
3,13518,13r9QY6cmjc,"i'm still a little confused, what is u naught ...",22. Diagonalization and Powers of A,"MIT 18.06 Linear Algebra, Spring 2005",1,train
4,9235,QVKj3LADCnA,"Thank you Sir, It is a great help to teach the...",2. Elimination with Matrices.,"MIT 18.06 Linear Algebra, Spring 2005",0,test
5,13146,PnPIqh7Frlw,hes using the same trousers since lecture 1,Lec 24: Simply connected regions; review | MIT...,"MIT 18.02 Multivariable Calculus, Fall 2007",0,test
6,8160,BSAA0akmPEU,Pen di siri calculus de andar poori dunya basi...,"Lec 9 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",0,train
7,8807,Z_5uLqcwDgM,"So, can someone explain (or point in the right...",Lecture 10: Survey of Difficulties with Ax = b,"MIT 18.065 Matrix Methods in Data Analysis, Si...",1,train
8,13488,4sTKcvYMNxk,i used the varibles f(x) and g(x) instead of u...,"Lec 4 | MIT 18.01 Single Variable Calculus, Fa...","MIT 18.01 Single Variable Calculus, Fall 2006",0,train
9,4808,Y4f7K9XF04k,How can I find the proof of Eckart-Young theor...,7. Eckart-Young: The Closest Rank k Matrix to A,"MIT 18.065 Matrix Methods in Data Analysis, Si...",1,train


## 10. Ajouter le contexte transcript

Pour améliorer le modèle, on peut joindre le commentaire avec un extrait du transcript de la même vidéo.

Pourquoi:
- le commentaire seul peut être ambigu
- le transcript apporte le contexte du cours
- le modèle comprend mieux de quelle notion parle l'étudiant

Comme on n'a pas d'alignement précis par timestamp, on prend un extrait simple du transcript associé à `video_id`.


In [14]:
transcripts_dir = base_dir / 'transcripts'
transcript_lookup = {}

for path in sorted(transcripts_dir.glob('*.txt')):
    try:
        transcript_lookup[path.stem] = path.read_text(encoding='utf-8', errors='ignore').strip()
    except Exception:
        transcript_lookup[path.stem] = ''

if 'binary_df' in globals():
    context_df = binary_df[['comment_id', 'video_id', 'comment_text', 'video_title', 'playlist_title', 'confusion_label', 'split']].copy()
    context_df['transcript_text'] = context_df['video_id'].map(transcript_lookup).fillna('')
    context_df['transcript_excerpt'] = context_df['transcript_text'].apply(lambda text: ' '.join(text.split()[:250]))
    context_df['text_with_context'] = context_df.apply(
        lambda row: f"{row['comment_text']} [SEP] {row['transcript_excerpt']}".strip(),
        axis=1,
    )

    context_path = base_dir / 'processed' / 'sight_bert_with_transcript_context.csv'
    context_df[['comment_id', 'video_id', 'text_with_context', 'comment_text', 'transcript_excerpt', 'video_title', 'playlist_title', 'confusion_label', 'split']].to_csv(context_path, index=False)

    print('Transcript context dataset created.')
    print('Rows:', len(context_df))
    print('Missing transcripts:', int((context_df['transcript_text'] == '').sum()))
    print('Saved file:')
    print(context_path)
    print('\nPreview:')
    display(context_df[['comment_id', 'video_id', 'text_with_context', 'confusion_label', 'split']].head(10))
else:
    print('binary_df not available')


Transcript context dataset created.
Rows: 15784
Missing transcripts: 0
Saved file:
C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_bert_with_transcript_context.csv

Preview:


,comment_id,video_id,text_with_context,confusion_label,split
0,7134,HgEqXhsIq_g,Jerison rocks! [SEP] The following content is ...,0,train
1,1270,9FLItlbBUPY,46:27 Top 10 saddest anime betrayals [SEP] The...,0,test
2,13959,ryLdyDrBfvI,"moises cañas, mira como en el mit si responden...",0,train
3,13518,13r9QY6cmjc,"i'm still a little confused, what is u naught ...",1,train
4,9235,QVKj3LADCnA,"Thank you Sir, It is a great help to teach the...",0,test
5,13146,PnPIqh7Frlw,hes using the same trousers since lecture 1 [S...,0,test
6,8160,BSAA0akmPEU,Pen di siri calculus de andar poori dunya basi...,0,train
7,8807,Z_5uLqcwDgM,"So, can someone explain (or point in the right...",1,train
8,13488,4sTKcvYMNxk,i used the varibles f(x) and g(x) instead of u...,0,train
9,4808,Y4f7K9XF04k,How can I find the proof of Eckart-Young theor...,1,train


## 11. Entraîner XLM-R pour la détection de confusion

Cette section fine-tune un classifieur binaire pour Smart Teacher.
Le premier essai utilise le texte des commentaires, et la variante avec contexte transcript peut etre activee sans changer le code.
Le modele est ensuite sauvegarde dans `dataset/sight-main/data/models/xlm_roberta_confusion`.


In [16]:
from pathlib import Path
import inspect
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

if "base_dir" not in globals():
    base_dir = Path('..').resolve()

processed_dir = base_dir / 'processed'
model_output_dir = base_dir / 'models' / 'xlm_roberta_confusion'
model_output_dir.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 256
USE_TRANSCRIPT_CONTEXT = True

plain_dataset_path = processed_dir / 'sight_bert_format.csv'
context_dataset_path = processed_dir / 'sight_bert_with_transcript_context.csv'
training_path = context_dataset_path if USE_TRANSCRIPT_CONTEXT and context_dataset_path.exists() else plain_dataset_path

if not training_path.exists():
    raise FileNotFoundError(f'Missing training file: {training_path}')

print('Base dir:', base_dir)
print('Training file:', training_path)
print('Model output dir:', model_output_dir)


Base dir: C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data
Training file: C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\processed\sight_bert_with_transcript_context.csv
Model output dir: C:\Users\Admin\Downloads\smart_teacher\smart_teacher\dataset\sight-main\data\models\xlm_roberta_confusion


In [17]:
raw_df = pd.read_csv(training_path)

if 'text_with_context' in raw_df.columns:
    text_column = 'text_with_context'
elif 'text' in raw_df.columns:
    text_column = 'text'
else:
    raise ValueError('Missing text column in training file.')

if 'label' in raw_df.columns:
    label_column = 'label'
elif 'confusion_label' in raw_df.columns:
    label_column = 'confusion_label'
else:
    raise ValueError('Missing label column in training file.')

training_df = raw_df[[text_column, label_column, 'split', 'comment_id', 'video_id']].copy()
training_df = training_df.rename(columns={text_column: 'text', label_column: 'label'})
training_df['text'] = training_df['text'].fillna('').astype(str).str.strip()
training_df = training_df[training_df['text'] != ''].copy()
training_df['label'] = training_df['label'].astype(int)

split_series = training_df['split'].astype(str).str.lower()
train_df = training_df[split_series == 'train'].reset_index(drop=True)
val_df = training_df[split_series.isin(['validation', 'val', 'dev'])].reset_index(drop=True)
test_df = training_df[split_series == 'test'].reset_index(drop=True)

print('Rows:', len(training_df))
print('Text column:', text_column)
print('Label column:', label_column)
print('Splits:', training_df['split'].value_counts().to_dict())
print('Label balance:', training_df['label'].value_counts().to_dict())
print('Train / Val / Test:', len(train_df), len(val_df), len(test_df))
display(training_df[['text', 'label', 'split']].head(10))


Rows: 15784
Text column: text_with_context
Label column: confusion_label
Splits: {'train': 11466, 'test': 2687, 'validation': 1631}
Label balance: {0: 13242, 1: 2542}
Train / Val / Test: 11466 1631 2687


,text,label,split
0,Jerison rocks! [SEP] The following content is ...,0,train
1,46:27 Top 10 saddest anime betrayals [SEP] The...,0,test
2,"moises cañas, mira como en el mit si responden...",0,train
3,"i'm still a little confused, what is u naught ...",1,train
4,"Thank you Sir, It is a great help to teach the...",0,test
5,hes using the same trousers since lecture 1 [S...,0,test
6,Pen di siri calculus de andar poori dunya basi...,0,train
7,"So, can someone explain (or point in the right...",1,train
8,i used the varibles f(x) and g(x) instead of u...,0,train
9,How can I find the proof of Eckart-Young theor...,1,train


In [18]:
class ConfusionTextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts = dataframe['text'].tolist()
        self.labels = dataframe['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item['labels'] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = ConfusionTextDataset(train_df, tokenizer, max_length=MAX_LENGTH)
validation_dataset = ConfusionTextDataset(val_df, tokenizer, max_length=MAX_LENGTH)
test_dataset = ConfusionTextDataset(test_df, tokenizer, max_length=MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

label_counts = train_df['label'].value_counts().to_dict()
total_train = len(train_df)
class_weights = torch.tensor(
    [
        total_train / (2.0 * label_counts.get(0, 1)),
        total_train / (2.0 * label_counts.get(1, 1)),
    ],
    dtype=torch.float32,
)
print('Class weights:', class_weights.tolist())


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary',
        zero_division=0,
    )
    metrics = {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }
    try:
        probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
        metrics['roc_auc'] = roc_auc_score(labels, probabilities[:, 1])
    except ValueError:
        metrics['roc_auc'] = float('nan')
    return metrics


class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


print('Train dataset size:', len(train_dataset))
print('Validation dataset size:', len(validation_dataset))
print('Test dataset size:', len(test_dataset))


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Class weights: [0.5961938500404358, 3.098918914794922]
Train dataset size: 11466
Validation dataset size: 1631
Test dataset size: 2687


In [22]:
import math

id2label = {0: 'not_confused', 1: 'confused'}
label2id = {'not_confused': 0, 'confused': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

training_args_kwargs = {
    'output_dir': str(model_output_dir / 'checkpoints'),
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'num_train_epochs': 2,
    'per_device_train_batch_size': 8 if torch.cuda.is_available() else 2,
    'per_device_eval_batch_size': 8 if torch.cuda.is_available() else 2,
    'gradient_accumulation_steps': 1 if torch.cuda.is_available() else 4,
    'logging_strategy': 'steps',
    'logging_steps': 50,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'greater_is_better': True,
    'report_to': 'none',
    'save_total_limit': 1,
    'fp16': torch.cuda.is_available(),
    'dataloader_num_workers': 0,
    'seed': 42,
}

effective_batch_size = training_args_kwargs['per_device_train_batch_size'] * training_args_kwargs['gradient_accumulation_steps']
steps_per_epoch = max(1, math.ceil(len(train_dataset) / effective_batch_size))
training_args_kwargs['warmup_steps'] = max(1, int(0.1 * steps_per_epoch * training_args_kwargs['num_train_epochs']))

training_signature = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in training_signature:
    training_args_kwargs['eval_strategy'] = 'epoch'
else:
    training_args_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': train_dataset,
    'eval_dataset': validation_dataset,
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}
trainer_signature = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in trainer_signature:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_signature:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = WeightedTrainer(
    class_weights=class_weights,
    **trainer_kwargs,
)

print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Training args ready.')


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5062.64it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cpu
Training args ready.


In [ ]:
train_result = trainer.train()

validation_metrics = trainer.evaluate(validation_dataset)
test_metrics = trainer.evaluate(test_dataset)

trainer.save_model(str(model_output_dir))
tokenizer.save_pretrained(str(model_output_dir))
trainer.save_state()

training_report = {
    'model_name': MODEL_NAME,
    'source_file': training_path.name,
    'text_column': text_column,
    'use_transcript_context': bool(training_path == context_dataset_path),
    'max_length': MAX_LENGTH,
    'train_rows': len(train_df),
    'validation_rows': len(val_df),
    'test_rows': len(test_df),
    'validation_metrics': validation_metrics,
    'test_metrics': test_metrics,
}
training_report_path = model_output_dir / 'training_report.json'
training_report_path.write_text(json.dumps(training_report, indent=2, ensure_ascii=False), encoding='utf-8')

print('Training finished.')
print('Model saved to:', model_output_dir)
print('Validation metrics:', validation_metrics)
print('Test metrics:', test_metrics)
print('Report saved to:', training_report_path)


In [23]:
def predict_confusion_score(text):
    trainer.model.eval()
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    inputs = {key: value.to(trainer.model.device) for key, value in inputs.items()}
    with torch.no_grad():
        logits = trainer.model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
    return {
        'not_confused': float(probabilities[0]),
        'confused': float(probabilities[1]),
    }

sample_text = 'I do not understand the last step, can you explain it again?'
print(predict_confusion_score(sample_text))


{'not_confused': 0.5953628420829773, 'confused': 0.4046371281147003}
